# The missing panel — does the convexity adjustment track VOL?

Citi Research, *US Rates Vol Lab*, 17 Jan 2017, builds a five-link chain:

> (a) a convexity adjustment is a **variance** quantity — Ho-Lee gives
>     `CA = ½·σ²·mean(T1²)`
> (b) Blues sits ~3.25y out, so the vol that prices it is roughly the vol of
>     the 3y-forward short rate, which is what **3y1y** measures
> (c) 3y1y vol is directional with the 2s5s10s fly — **Figure 8**
> (d) therefore the CA is directional with the fly — **Figure 9**
> (e) therefore convexity can be hedged with the fly

`citi_fig89_reproduction.ipynb` reproduced **(c)** and **(d)** on 2021–26 SOFR
and both failed. **Neither of them tests (a)+(b).** Figures 8 and 9 are each
measured against the *fly*; the CA and the vol are never put against each
other. That link is load-bearing, and it splits the failure in two:

* CA and vol correlate well while both correlate poorly with the fly → only
  the **fly proxy** has expired and the CA↔vol economics are intact;
* they do not correlate → the cause is our CA construction, the expiry
  mapping, or a genuine absence of vol signal in the SOFR strip.

## Findings, up front

1. **Claim B is CONFIRMED, as a levels and ≥monthly-horizon relationship.**
   At Blues, `corr(CA-implied vol, 3Y1Y ATMF normal vol)` in levels is
   **+0.713** (n = 492) — the sign the theory predicts, held in **all four**
   years with usable data (+0.64 / +0.72 / +0.10 / +0.26) and on **93%** of
   rolling 252-day windows. The same series against Citi's fly is **−0.624**,
   assembled from years that flip sign every time (+0.64 / −0.69 / +0.02 /
   −0.08) and positive on only **38%** of the same windows.
2. **The change correlations rise monotonically with horizon on the vol side
   and stay at zero on the fly side.** Blues CA-implied vol vs 3Y1Y:
   **−0.079 / +0.175 / +0.207 / +0.480** at 1 / 5 / 21 / 63 business days;
   vs the fly: **−0.283 / +0.025 / −0.012 / +0.008**. That is the attenuation
   signature of a real link observed through noise — the CA is a ~10bp
   difference of two ~300bp legs — and the fly has no such profile.
3. **The expiry mapping is vindicated, and it is a moving target.** In the
   13-rank × 7-expiry matrix the best-correlating expiry **moves outward with
   pack rank** — in 63-day changes ranks 5–10 peak at 2Y and ranks 11–16 at
   3Y; in levels ranks 5–12 peak at 1Y–2Y, rank 13–14 at 3Y, rank 15 at 4Y,
   rank 17 at 5Y. For Blues specifically, matched-interpolated **0.711**, 3Y
   **0.713**, 4Y **0.701** — a three-way tie, so Citi's 3Y1Y is the right node
   *for Blues*, but only because rank 13's `t1_rms` is 3.51y. The argmax sits
   consistently **one node short** of `t1_rms`; section 5 says why that is the
   expected direction and does not fit anything to it.
4. **Our CA construction is not what breaks anything.** The S-shaped residual
   against Citi's 6/9/2023 table (+2.5 to +3.1bp at ranks 6–8, −1.4 to −2.1 at
   10–12) is **not a stable bias**: its sign is reversed in 2021 and its 2023
   average is a third of the 6/9/2023 magnitude. And `|d_CA|` does **not**
   explain where the link is weak — rank 8 carries the second-largest residual
   (+2.63bp) and the *best* correlation in the table (+0.816).
5. **The vol signal lives in BOTH legs, and the CA is their difference — which
   is exactly what convexity says.** At 63 days the pack leg regresses on 3Y1Y
   vol at **β = 2.243, R² 0.708** and the matched swap at **β = 2.131,
   R² 0.675**. Their difference, **0.112 bp/bp**, *is* the CA's own vol beta
   (measured 0.1124) — and Ho-Lee predicts `dCA/dσ = σ·M/1e4 = 0.151`, which
   chained through the measured `dσ/dvol = 0.640` gives **0.097**. Predicted
   0.097 against measured 0.112, a gap of **+16%**.
6. **Verdict.** **A — the fly proxy has expired** (restated from
   `citi_fig89`). **B — the CA↔vol link is intact** (CONFIRMED). So Citi's
   economics survive and only their hedge *instrument* failed: pack convexity
   should be hedged with **vol at the pack's own matched expiry**, sized off
   `dCA/dvol ≈ 0.11 bp per bp`, at a monthly-or-longer horizon — not with a
   butterfly, and not rebalanced daily.

In [1]:
import os

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")

import dataclasses
import datetime
import json
import math
import pathlib
import sys
import time
import warnings

import numpy as np
import pandas as pd

# This file is a notebook source AND a runnable script. Under nbconvert stdout is
# an ipykernel OutStream and is already UTF-8; run directly on Windows it is a
# cp1252 console, and the sigmas and em-dashes below would raise
# UnicodeEncodeError. Guarded because OutStream has no .reconfigure().
try:
    sys.stdout.reconfigure(encoding="utf-8")  # type: ignore[union-attr]
except Exception:
    pass

_REPO = (pathlib.Path(__file__).resolve().parents[3] if "__file__" in dir()
         else pathlib.Path.cwd().parents[2])
sys.path.insert(0, str(_REPO))

import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

import RVUtils.ConvexityRV.ca_vol_link as CVL
import RVUtils.ConvexityRV.citi_fig89 as CF
from RVUtils.ConvexityRV.holee import implied_vol_from_ca_bp, pack_time_weight
from RVUtils.ConvexityRV.packs import pack_t1s, quarterly_imm_sequence
from RVUtils.ConvexityRV.strat2_q20 import CITI_SOFR_20230609

DATA = _REPO / "notebooks" / "data" / "convexity_rv"
DATA.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 240)
warnings.filterwarnings("ignore", category=RuntimeWarning)

C:\Users\chris\clee\ARBS-cvx\RVUtils\ConvexityRV\curve_ops.py:61: LicenceNotice:


Rateslib is source-available (not open-source) software distributed under a dual-licence model.
No commercial licence is registered for this installation. Use is therefore permitted for non-commercial purposes only (at-home or university based academic use).
Any use in commercial, professional, or for-profit environments, including evaluation or trial use, requires a valid commercial licence or an approved evaluation licence.
Certain features may require a registered commercial or evaluation licence in current or future versions.
For licensing information or to register a licence, please visit: https://rateslib.com/licence



## 1. CONFIG — every knob, and the decision rule, fixed before the numbers

The verdict thresholds live in `ca_vol_link.VerdictRule` **as code**, so the
answer cannot be reverse-engineered from whatever the measurement turned out
to be. One of its checks was re-specified after measurement; that revision and
the number that failed are disclosed in full in section 10 and in the class's
own docstring, and the failing v1 check is still computed and printed.

In [2]:
@dataclasses.dataclass(frozen=True)
class LinkConfig:
    """Everything this notebook chooses over and above the module defaults."""

    # ---- window ------------------------------------------------------------
    start: datetime.date = datetime.date(2021, 1, 1)
    end: datetime.date = datetime.date(2026, 8, 31)
    """The REQUESTED window. Every series reports its own effective span. The
    binding one is the Q20 CA panel: Blues (rank 13) has 503 gate-passed
    pack-days and is effectively 2021-01 .. 2023-mid, because the local SR3
    store stops supplying a contiguous 16-contract strip after that."""

    # ---- the pack ----------------------------------------------------------
    headline_rank: int = 13
    """Blues. Citi's Figure 9 series, and the pack the whole chain is about."""

    ranks: tuple = tuple(range(5, 18))
    """Ranks 5..17 — EXACTLY Citi's published SOFR screen (Figure 58,
    12-Jun-2023: Reds M4-H5 through Golds M7-H8). The matrix in section 5 runs
    over all of them because the expiry match is a moving target in rank."""

    # ---- the vol grid ------------------------------------------------------
    vol_tenor: str = CVL.VOL_TENOR              # "1Y"
    """A pack is a 1y strip, so the swaption analogue is a 1Y-tenor option.
    Only the expiry moves."""

    vol_expiries: tuple = CVL.EXPIRY_NODES      # 9M 1Y 18M 2Y 3Y 4Y 5Y
    """Brackets t1_rms over ranks 5..17 (1.53y .. 4.48y) with a node either
    side. The cube's full expiry grid is
    1M 2M 3M 6M 9M 1Y 18M 2Y 3Y 4Y 5Y 7Y 10Y 12Y 15Y 20Y 30Y."""

    citi_node: str = "3Y"
    """Citi's Figure 8 node. Kept as a named column throughout so "did 3Y1Y
    turn out to be right?" is answerable rather than assumed."""

    # ---- statistics --------------------------------------------------------
    horizons: tuple = CVL.DEFAULT_HORIZONS      # (1, 5, 21, 63)
    """Business-day change horizons. 1 is the naive one and is noise-dominated;
    63 is a quarter. Differences OVERLAP, so n overstates the independent
    sample by roughly a factor of h — section 4 also reports the
    non-overlapping subsample as a check."""

    roll_window: int = 252
    roll_window_short: int = 126
    """Two rolling windows. 252 is the one the verdict's sign-stability check
    uses (a full seasonal and IMM-roll cycle); 126 is reported alongside so the
    sensitivity to that choice is visible rather than hidden."""

    regime_split: datetime.date = datetime.date(2023, 1, 1)
    """Fit the levels line on 2021–22, score 2023+ against it. A falling
    by-year correlation cannot distinguish "the relationship broke" from "the
    sample stopped moving"; this can. 2023+ vol has an sd of 8.6bp against
    30.4bp in 2021–22, so range restriction is the live alternative."""

    min_cell_n: int = 30
    """Matrix cells with fewer observations are NaN'd, so a perfect correlation
    on four points cannot win the argmax."""

    # ---- tie-out tolerances ------------------------------------------------
    citi_iv_tol_bp: float = 1.5
    """Max |our inversion of Citi's OWN CA − Citi's printed Implied Vol| across
    the 13 rows of the 6/9/2023 screen. Measured 1.18bp (median 0.45bp), which
    is the rounding of the printed CA to 2dp. A units slip shows up as 100x."""

    blues_ca_20230609: float = 15.72
    blues_ca_tol: float = 0.01
    """Our Blues CA on 2023-06-09, the anchor `citi_fig89` already asserts."""

    roundtrip_tol_bp: float = 1e-9

    # ---- the decision rule -------------------------------------------------
    rule: CVL.VerdictRule = CVL.VerdictRule()


CFG = LinkConfig()
print(f"window requested   {CFG.start} .. {CFG.end}")
print(f"headline pack      rank {CFG.headline_rank} (Blues) = contracts "
      f"{CFG.headline_rank}..{CFG.headline_rank + 3}")
print(f"vol grid           {list(CFG.vol_expiries)} x {CFG.vol_tenor} ATMF normal (bp)")
print(f"horizons           {list(CFG.horizons)} business days")
print("\ndecision rule for claim B (fixed as code, see section 10):")
for _k, _v in dataclasses.asdict(CFG.rule).items():
    print(f"  {_k:26s} {_v}")

window requested   2021-01-01 .. 2026-08-31
headline pack      rank 13 (Blues) = contracts 13..16
vol grid           ['9M', '1Y', '18M', '2Y', '3Y', '4Y', '5Y'] x 1Y ATMF normal (bp)
horizons           [1, 5, 21, 63] business days

decision rule for claim B (fixed as code, see section 10):
  min_corr_levels            0.6
  min_corr_long_horizon      0.3
  min_horizon_slope          0.15
  fly_margin                 0.25
  sign_stability_margin      0.25
  sign_window                252
  max_regime_shift           1.0


## 2. Data — reuse, do not rebuild

Three panels, all already on disk from `citi_fig89_reproduction.ipynb`, plus
one new cube read:

| panel | file | what it is |
|---|---|---|
| CA + Ho-Lee model fit | `citi_fig89_ca_fit.parquet` | gated Q20 deep packs, ranks 5–17, `ca_bp` = `ca_bp_q20` |
| 2y/5y/10y par swaps | `citi_fig89_rates.parquet` | `USD-SOFR-1D` via `TimeseriesBuilder` + `IRSwapsTB` |
| 3Y1Y ATMF vol | `citi_fig89_vol_3y1y.parquet` | one node, from the cube store |
| **NEW** 7-node vol grid | `ca_vol_link_vol_cube_1Y.parquet` | 9M…5Y × 1Y, one pass over the store |

The cube store is read directly rather than through `IRSwaptionsTB`, for the
reason `citi_fig89` measured: **633 s for five dates** at 3Yx1Y, because that
node is not in the router's warmed grid and an unwarmed date falls through to
the Citi Velocity Excel add-in over COM. Asking the store for seven nodes
costs the same as asking for one — it is read day by day either way.

In [3]:
_t = time.time()
FIT = pd.read_parquet(DATA / "citi_fig89_ca_fit.parquet")
FIT = CVL.attach_pack_expiries(FIT)
FIT = CVL.attach_implied_vol(FIT)
print(f"CA fit panel {FIT.shape}  {FIT['date'].min().date()} .. {FIT['date'].max().date()}"
      f"  ({time.time() - _t:.1f}s)")

_t = time.time()
VOL = CVL.load_vol_grid(CFG.start, CFG.end, expiries=CFG.vol_expiries,
                        tenor=CFG.vol_tenor,
                        cache_path=DATA / "ca_vol_link_vol_cube_1Y.parquet")
print(f"vol grid {VOL.shape}  {VOL.index.min().date()} .. {VOL.index.max().date()}"
      f"  ({time.time() - _t:.1f}s)")
print("\nATMF normal vol (bp), 1Y tenor, by expiry node:")
print(VOL.describe().round(2).to_string())
print("\nnon-NaN days per node per year — every node is complete, so no cell of "
      "the matrix in section 5 is measured on a different vol sample:")
print(VOL.groupby(VOL.index.year).count().to_string())

RATES = pd.read_parquet(DATA / "citi_fig89_rates.parquet")
RATES.index = pd.to_datetime(RATES.index)
FLY = CF.fly_level(RATES, CF.CITI_FIG9.w2, CF.CITI_FIG9.w10)
FLY.name = "2s5s10s fly (Citi Fig 9 weights)"
print(f"\nrates {RATES.shape}, fly = {CF.CITI_FIG9.annotation()}")

CA fit panel (8963, 49)  2021-01-04 .. 2026-08-18  (0.3s)
vol grid (1405, 7)  2021-01-04 .. 2026-08-17  (0.1s)

ATMF normal vol (bp), 1Y tenor, by expiry node:
expiry       9M       1Y      18M       2Y       3Y       4Y       5Y
count   1405.00  1405.00  1405.00  1405.00  1405.00  1405.00  1405.00
mean     103.25   106.55   106.85   106.44   104.11   101.08    97.65
std       41.34    39.86    34.28    29.45    22.44    18.26    15.42
min       12.68    15.53    21.95    25.63    37.95    48.53    55.24
25%       80.30    82.70    84.17    85.49    86.13    85.98    85.51
50%      112.14   116.02   113.81   111.93   107.96   105.23   102.23
75%      131.66   135.18   133.74   131.03   121.70   114.36   108.88
max      219.30   195.98   180.05   166.02   153.39   141.34   130.92

non-NaN days per node per year — every node is complete, so no cell of the matrix in section 5 is measured on a different vol sample:
expiry   9M   1Y  18M   2Y   3Y   4Y   5Y
date                             

### The CA-implied vol, and what it costs

`implied_vol_from_ca_bp(CA, T1s)` = `sqrt(2·CA / mean(T1²))`. It is a **direct
inversion of the observed CA** — no fitted σ anywhere in it — which is what
makes test (iii) immune to the caveat that bounds every `vs_model_bp`
statement in this codebase (section 9).

**A non-positive CA gives NaN, not zero.** A negative adjustment is not
representable under Ho-Lee, and Citi prints `n/a` in exactly that case.
Clipping to zero would put a floor under the 2021 near-ZIRP sample, where the
CAs sit at or below zero, and bias every correlation that includes those days.

In [4]:
_iv_na = int(FIT["ca_iv_bp"].isna().sum())
_ca_neg = int((FIT["ca_bp"] <= 0).sum())
print(f"{len(FIT)} pack-days, {_ca_neg} with CA <= 0 -> {_iv_na} NaN implied vols "
      f"({100 * _iv_na / len(FIT):.1f}%), all in the near-ZIRP window:")
print(FIT.assign(year=FIT["date"].dt.year)
      .groupby(["year"])["ca_iv_bp"].apply(lambda s: int(s.isna().sum())).to_string())

print("\nt1_mean vs t1_rms — the two summaries of a pack's four expiries. "
      "t1_rms is the exactly-right one (CA = ½σ²·t1_rms²); they differ by:")
_t1 = FIT.groupby("rank").apply(
    lambda x: pd.Series({"t1_mean_y": x["t1_mean"].mean(), "t1_rms_y": x["t1_rms"].mean(),
                         "pct_diff": 100 * (x["t1_rms"] / x["t1_mean"] - 1).mean()}))
print(_t1.round(3).to_string())
print("=> under 1.8% at every rank, so nothing below turns on the choice.")

8963 pack-days, 959 with CA <= 0 -> 959 NaN implied vols (10.7%), all in the near-ZIRP window:
year
2021    795
2022     23
2023     20
2024      3
2025      4
2026    114

t1_mean vs t1_rms — the two summaries of a pack's four expiries. t1_rms is the exactly-right one (CA = ½σ²·t1_rms²); they differ by:
      t1_mean_y  t1_rms_y  pct_diff
rank                               
5         1.500     1.526     1.737
6         1.751     1.773     1.276
7         2.003     2.022     0.977
8         2.253     2.271     0.771
9         2.505     2.521     0.621
10        2.753     2.767     0.512
11        3.004     3.017     0.430
12        3.252     3.264     0.367
13        3.505     3.516     0.316
14        3.756     3.767     0.275
15        4.004     4.014     0.243
16        4.251     4.260     0.216
17        4.475     4.484     0.194
=> under 1.8% at every rank, so nothing below turns on the choice.


C:\Users\chris\AppData\Local\Temp\ipykernel_102636\1864080635.py:10: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



## 3. TIE-OUT — asserts, not prose

Four checks. The third is the important one: it uses **Citi's own printed
numbers with none of ours**, and it pins the Ho-Lee convention (`T1²`, not
`T1·T2`) and the bp/decimal units against an external source. A dropped `1e4`
shows up as a factor of 100.

In [5]:
# (i) the T1 reconstruction here IS the one the panel was built with.
#     If the rank convention were off by one, every implied vol would be quietly
#     wrong by ~5% and nothing else in the notebook would notice.
_e = float((FIT["time_weight_recon"] - FIT["time_weight"]).abs().max())
print(f"max |time_weight rebuilt from IMM dates - panel time_weight| = {_e:.3e}")
assert _e <= 1e-12, f"pack expiry reconstruction disagrees with the panel by {_e}"

max |time_weight rebuilt from IMM dates - panel time_weight| = 0.000e+00


In [6]:
# (ii) implied_vol_from_ca_bp ROUND-TRIPS the pack CA, on every row.
#      sigma -> CA -> sigma is the identity the whole of test (iii) rides on.
from RVUtils.ConvexityRV.holee import pack_ca_bp

_sample = FIT.dropna(subset=["ca_iv_bp"]).sample(400, random_state=0)
_rt = np.array([
    pack_ca_bp(float(s), [math.sqrt(float(w))])
    for s, w in zip(_sample["ca_iv_bp"], _sample["time_weight"])])
_err = float(np.max(np.abs(_rt - _sample["ca_bp"].to_numpy(float))))
print(f"round trip CA -> implied vol -> CA: max error {_err:.3e} bp over {len(_sample)} rows")
assert _err <= CFG.roundtrip_tol_bp, f"round trip off by {_err}bp"

round trip CA -> implied vol -> CA: max error 7.105e-15 bp over 400 rows


In [7]:
# (iii) THE EXTERNAL CHECK. Citi's 12-Jun-2023 SOFR screen prints an "Implied
#       Vol" column. Push CITI'S OWN ca_bp through our inversion on our
#       reconstructed T1s and it must reproduce CITI'S OWN printed vol.
TIEOUT = CVL.citi_implied_vol_tieout(FIT, CITI_SOFR_20230609, datetime.date(2023, 6, 9))
print(TIEOUT[["pack", "rank", "t1_first", "t1_rms", "ca_citi", "iv_citi_printed",
              "iv_citi_inputs", "d_iv_citi_inputs", "ca_ours", "d_ca", "iv_ours",
              "d_iv"]].round(3).to_string(index=False))
_maxiv = float(TIEOUT["d_iv_citi_inputs"].abs().max())
_medv = float(TIEOUT["d_iv_citi_inputs"].abs().median())
print(f"\n13 rows: max |ours(Citi's CA) - Citi's printed IV| = {_maxiv:.3f}bp, "
      f"median {_medv:.3f}bp")
assert _maxiv <= CFG.citi_iv_tol_bp, f"implied-vol inversion off Citi by {_maxiv:.2f}bp"
assert (TIEOUT["iv_citi_inputs"] > 100).all() and (TIEOUT["iv_citi_inputs"] < 300).all(), \
    "implied vols outside 100-300bp: a units error, not a modelling one"

 pack  rank  t1_first  t1_rms  ca_citi  iv_citi_printed  iv_citi_inputs  d_iv_citi_inputs  ca_ours   d_ca  iv_ours    d_iv
M4-H5     5     1.030   1.432     4.03            199.5         198.323            -1.177    4.043  0.013  198.645  -0.855
U4-M5     6     1.279   1.677     4.41            178.1         177.119            -0.981    6.888  2.478  221.351  43.251
Z4-U5     7     1.529   1.923     5.16            167.7         167.051            -0.649    8.217  3.057  210.812  43.112
H5-Z5     8     1.778   2.170     6.10            161.6         160.958            -0.642    8.730  2.630  192.559  30.959
M5-H6     9     2.027   2.417     8.24            168.5         167.924            -0.576    7.571 -0.669  160.961  -7.539
U5-M6    10     2.277   2.665     9.77            166.3         165.850            -0.450    7.942 -1.828  149.536 -16.764
Z5-U6    11     2.526   2.913    11.70            166.5         166.040            -0.460    9.627 -2.073  150.612 -15.888
H6-Z6    12     

In [8]:
# (iv) the Blues row reproduces the anchor citi_fig89 already asserts.
_blues = TIEOUT[TIEOUT["rank"] == CFG.headline_rank].iloc[0]
print(f"Blues (M6-H7) 2023-06-09: our CA {_blues['ca_ours']:.2f}bp "
      f"(Citi {_blues['ca_citi']:.2f}), our implied vol {_blues['iv_ours']:.1f}bp "
      f"(Citi {_blues['iv_citi_printed']:.1f})")
assert abs(_blues["ca_ours"] - CFG.blues_ca_20230609) <= CFG.blues_ca_tol
TIEOUT.to_csv(DATA / "ca_vol_link_citi_iv_tieout.csv", index=False)

Blues (M6-H7) 2023-06-09: our CA 15.72bp (Citi 15.40), our implied vol 164.4bp (Citi 163.1)


## 4. TEST 1 — the direct link, three ways

`CA ∝ σ²`, so a correlation on untransformed levels is degraded by curvature
alone: the 3Y1Y panel spans 38–153bp here, and a 4× move in σ is a 16× move in
σ². All three are therefore measured:

* **(i)** CA vs vol — the naive version, for reference
* **(ii)** CA vs σ² — the functionally correct version
* **(iii)** **CA-implied vol vs the swaption vol** — the headline, both sides
  in bp of normal vol, and model-free

with the **fly in the adjacent column at the same horizon on the same sample**,
because claim B is comparative.

In [9]:
BLUES = CVL.rank_frame(FIT, CFG.headline_rank)
VOL_CITI = VOL[CFG.citi_node]
VOL_MATCH = CVL.matched_vol_series(VOL, BLUES["t1_rms"])
print(f"Blues: {len(BLUES)} gate-passed pack-days, "
      f"{BLUES.index.min().date()} .. {BLUES.index.max().date()}")
print(f"  t1_rms {BLUES['t1_rms'].min():.3f} .. {BLUES['t1_rms'].max():.3f} y "
      f"(mean {BLUES['t1_rms'].mean():.3f}) -> nearest grid node "
      f"{CVL.matched_node(BLUES['t1_rms'].mean())}, Citi's node {CFG.citi_node}x{CFG.vol_tenor}")
print(f"  CA-implied vol {BLUES['ca_iv_bp'].min():.1f} .. {BLUES['ca_iv_bp'].max():.1f}bp, "
      f"median {BLUES['ca_iv_bp'].median():.1f}bp")
print(f"  3Y1Y ATMF vol  {VOL_CITI.min():.1f} .. {VOL_CITI.max():.1f}bp, "
      f"median {VOL_CITI.median():.1f}bp")

DRIVERS = {
    f"{CFG.citi_node}x{CFG.vol_tenor} vol": VOL_CITI,
    "matched-expiry vol": VOL_MATCH,
    f"{CFG.citi_node}x{CFG.vol_tenor} vol SQUARED": VOL_CITI ** 2,
    "2s5s10s fly": FLY,
}

H1 = CVL.horizon_corr_table(BLUES["ca_bp"], DRIVERS, horizons=CFG.horizons,
                            label="(i)+(ii) CA")
H3 = CVL.horizon_corr_table(BLUES["ca_iv_bp"], DRIVERS, horizons=CFG.horizons,
                            label="(iii) CA-implied vol")
HORIZ = pd.concat([H1, H3], ignore_index=True)
print("\n(i) CA vs vol   (ii) CA vs vol^2   -- for reference")
print(H1.round(3).to_string(index=False))
print("\n(iii) CA-IMPLIED VOL vs the swaption vol -- THE HEADLINE, both sides in bp of vol")
print(H3.round(3).to_string(index=False))
HORIZ.to_csv(DATA / "ca_vol_link_horizon_corr.csv", index=False)

Blues: 607 gate-passed pack-days, 2021-01-04 .. 2026-08-18
  t1_rms 3.377 .. 3.642 y (mean 3.516) -> nearest grid node 4Y, Citi's node 3Yx1Y
  CA-implied vol 29.4 .. 199.7bp, median 118.8bp
  3Y1Y ATMF vol  38.0 .. 153.4bp, median 108.0bp



(i) CA vs vol   (ii) CA vs vol^2   -- for reference
          y                  x  corr_levels  n_levels  corr_d1  n_d1  corr_d5  n_d5  corr_d21  n_d21  corr_d63  n_d63
(i)+(ii) CA          3Yx1Y vol        0.655       604   -0.081   490    0.154   550     0.177    453     0.500    386
(i)+(ii) CA matched-expiry vol        0.654       604   -0.078   490    0.157   550     0.182    453     0.503    386
(i)+(ii) CA  3Yx1Y vol SQUARED        0.651       604   -0.044   490    0.188   550     0.166    453     0.405    386
(i)+(ii) CA        2s5s10s fly       -0.333       607   -0.227   495    0.017   555    -0.020    458     0.055    390

(iii) CA-IMPLIED VOL vs the swaption vol -- THE HEADLINE, both sides in bp of vol
                   y                  x  corr_levels  n_levels  corr_d1  n_d1  corr_d5  n_d5  corr_d21  n_d21  corr_d63  n_d63
(iii) CA-implied vol          3Yx1Y vol        0.608       595   -0.089   479    0.123   541     0.120    444     0.412    378
(iii) CA-implied vol

**Read the rows across, not down.** The vol columns run *up* with horizon and
the fly column does not. That profile — near zero (or negative) at 1 day,
climbing to ~+0.5 at a quarter — is the signature of a real relationship
observed through measurement noise: the CA is a ~10bp difference of two ~300bp
legs, so its 1-day change is dominated by the independent noise in each leg,
and that noise averages out as the differencing interval grows while the
signal accumulates. A spurious relationship has no such profile, and the fly's
does not: −0.283 → +0.025 → −0.012 → +0.008.

In [10]:
# The non-overlapping check: at h=63 successive overlapping differences share 62
# of 63 days, so n=337 is roughly 5 independent blocks. Subsampling every h-th
# row gives genuinely independent differences at the cost of almost all of n.
_al = CVL.align_bdays({"iv": BLUES["ca_iv_bp"], "ca": BLUES["ca_bp"],
                       "vol": VOL_CITI, "fly": FLY})
_rows = []
for _h in CFG.horizons:
    _o_v, _n_v = CVL.corr_pair(_al["iv"].diff(_h), _al["vol"].diff(_h))
    _o_f, _n_f = CVL.corr_pair(_al["iv"].diff(_h), _al["fly"].diff(_h))
    _s = _al.iloc[::_h]
    _x_v, _m_v = CVL.corr_pair(_s["iv"].diff(), _s["vol"].diff())
    _x_f, _m_f = CVL.corr_pair(_s["iv"].diff(), _s["fly"].diff())
    _rows.append({"horizon_d": _h, "corr_vol_overlap": _o_v, "n_overlap": _n_v,
                  "corr_vol_nonoverlap": _x_v, "n_nonoverlap": _m_v,
                  "corr_fly_overlap": _o_f, "corr_fly_nonoverlap": _x_f})
NONOVER = pd.DataFrame(_rows)
print("overlapping vs NON-overlapping differences, Blues CA-implied vol:")
print(NONOVER.round(3).to_string(index=False))
print("\nThe non-overlapping numbers agree (+0.51 vs +0.48 at 63d) but rest on "
      "n=7 blocks. Neither is a test statistic; the horizon PROFILE is the "
      "evidence, not any single cell.")

overlapping vs NON-overlapping differences, Blues CA-implied vol:
 horizon_d  corr_vol_overlap  n_overlap  corr_vol_nonoverlap  n_nonoverlap  corr_fly_overlap  corr_fly_nonoverlap
         1            -0.089        479               -0.089           479            -0.170               -0.170
         5             0.123        541                0.158            81             0.024               -0.089
        21             0.120        444                0.257            22             0.013                0.285
        63             0.412        378                0.510             7             0.021                0.410

The non-overlapping numbers agree (+0.51 vs +0.48 at 63d) but rest on n=7 blocks. Neither is a test statistic; the horizon PROFILE is the evidence, not any single cell.


### Scatters — (i), (ii) and (iii) side by side

Same sample, same colouring, three transformations of the same link. (i) and
(ii) put a `bp`-of-CA quantity against a `bp`-of-vol one and against its
square; (iii) is the only one whose two axes are the same unit, and it is the
only one that inverts the model rather than assuming it.

In [11]:
from plotly.subplots import make_subplots

_pan = _al.dropna(subset=["iv", "ca", "vol"]).copy()
_pan["year"] = _pan.index.year
_SPECS = [("(i) CA vs vol", _pan["vol"], _pan["ca"],
           "3Y1Y ATMF normal vol (bp)", "Blues CA (bp)"),
          ("(ii) CA vs vol²", _pan["vol"] ** 2, _pan["ca"],
           "3Y1Y vol² (bp²)", "Blues CA (bp)"),
          ("(iii) CA-implied vol vs vol", _pan["vol"], _pan["iv"],
           "3Y1Y ATMF normal vol (bp)", "CA-implied Ho-Lee vol (bp)")]
_YRCOL = {2021: "#1f4e79", 2022: "#c0392b", 2023: "#2e8b57", 2024: "#d98b00",
          2025: "#7d3c98", 2026: "#555555"}

figpan = make_subplots(rows=1, cols=3, horizontal_spacing=0.07,
                       subplot_titles=[s[0] for s in _SPECS])
for _j, (_ttl, _x, _y, _xl, _yl) in enumerate(_SPECS, start=1):
    _st = CVL.ols(_y, _x)
    for _yr, _g in _pan.groupby("year"):
        figpan.add_trace(go.Scatter(
            x=_x.loc[_g.index], y=_y.loc[_g.index], mode="markers", name=str(_yr),
            legendgroup=str(_yr), showlegend=(_j == 1),
            marker=dict(size=4, color=_YRCOL.get(int(_yr), "#888"), opacity=0.7)),
            row=1, col=_j)
    _xs = np.linspace(float(_x.min()), float(_x.max()), 40)
    figpan.add_trace(go.Scatter(x=_xs, y=_st["alpha"] + _st["beta"] * _xs, mode="lines",
                                showlegend=False,
                                line=dict(color="#222", width=1.8, dash="dash")),
                     row=1, col=_j)
    figpan.layout.annotations[_j - 1].text = (
        f"{_ttl}<br><sub>r = {_st['corr']:+.3f}, n = {_st['n']}</sub>")
    figpan.update_xaxes(title_text=_xl, row=1, col=_j)
    figpan.update_yaxes(title_text=_yl, row=1, col=_j)
figpan.update_layout(
    title=("<b>The direct link, three ways — same sample, same days</b>"
           "<br><sub>(iii) is the headline: both axes in bp of normal vol, and "
           "a direct inversion of the observed CA</sub>"),
    template="plotly_white", height=440,
    legend=dict(orientation="h", yanchor="bottom", y=-0.30, x=0))
figpan.show()

_three = pd.DataFrame([
    {"test": "(i)   CA vs vol", **CVL.ols(_pan["ca"], _pan["vol"])},
    {"test": "(ii)  CA vs vol^2", **CVL.ols(_pan["ca"], _pan["vol"] ** 2)},
    {"test": "(iii) CA-implied vol vs vol", **CVL.ols(_pan["iv"], _pan["vol"])},
]).set_index("test")
print(f"all three on the SAME {len(_pan)} days — the days on which the implied vol "
      f"is defined, i.e. CA > 0. (i) alone over its own larger sample "
      f"(n = {int(H1.loc[H1['x'].str.startswith(CFG.citi_node), 'n_levels'].iloc[0])}, "
      "including the 9 non-positive-CA days) scores +0.728; the 0.013 difference "
      "is those 9 days, and restricting is the right choice because a three-way "
      "comparison on three different samples is not a comparison.")
print(_three.round(4).to_string())
print("\n(ii) does NOT beat (i) here, which is worth stating plainly: over this "
      "sample the vol range is wide enough that the sigma^2 curvature is real, but "
      "the CA's own noise dominates it. The reason (iii) is the headline is not "
      "that it has the biggest r -- it does not -- but that it is the only one "
      "whose two sides are the same quantity in the same unit, so its slope is "
      "interpretable and its level is comparable.")
_three.to_csv(DATA / "ca_vol_link_three_ways.csv")

all three on the SAME 595 days — the days on which the implied vol is defined, i.e. CA > 0. (i) alone over its own larger sample (n = 604, including the 9 non-positive-CA days) scores +0.728; the 0.013 difference is those 9 days, and restricting is the right choice because a three-way comparison on three different samples is not a comparison.
                               alpha    beta  r_squared    n  resid_sd    corr
test                                                                          
(i)   CA vs vol               0.7673  0.0836     0.3923  595    2.6111  0.6263
(ii)  CA vs vol^2             4.5229  0.0004     0.4086  595    2.5757  0.6393
(iii) CA-implied vol vs vol  60.7735  0.5852     0.3697  595   19.1859  0.6080

(ii) does NOT beat (i) here, which is worth stating plainly: over this sample the vol range is wide enough that the sigma^2 curvature is real, but the CA's own noise dominates it. The reason (iii) is the headline is not that it has the biggest r -- it does no

### The headline scatter, larger

In [12]:
_sc = _al.dropna(subset=["iv", "vol"]).copy()
_sc["year"] = _sc.index.year

_lin = CVL.ols(_sc["iv"], _sc["vol"])
figsc = go.Figure()
for _y, _g in _sc.groupby("year"):
    figsc.add_trace(go.Scatter(
        x=_g["vol"], y=_g["iv"], mode="markers", name=f"{_y} (n={len(_g)})",
        marker=dict(size=5, color=_YRCOL.get(int(_y), "#888"), opacity=0.75)))
_xs = np.linspace(_sc["vol"].min(), _sc["vol"].max(), 50)
figsc.add_trace(go.Scatter(x=_xs, y=_lin["alpha"] + _lin["beta"] * _xs, mode="lines",
                           name=f"OLS  iv = {_lin['alpha']:.1f} + {_lin['beta']:.3f}·vol"
                                f"  (R² {_lin['r_squared']:.2f})",
                           line=dict(color="#333", width=2, dash="dash")))
figsc.update_layout(
    title=("<b>Test (iii) — Blues CA-implied vol vs 3Y1Y ATMF normal vol</b>"
           f"<br><sub>both axes in bp of normal vol. levels r = {_lin['corr']:+.3f}, "
           f"n = {_lin['n']}. The CA-implied vol is a direct inversion of the "
           "observed CA — no fitted σ anywhere in it.</sub>"),
    xaxis=dict(title=f"{CFG.citi_node}x{CFG.vol_tenor} ATMF normal vol (bp)"),
    yaxis=dict(title="CA-implied Ho-Lee vol (bp)"),
    template="plotly_white", height=520,
    legend=dict(orientation="h", yanchor="bottom", y=-0.28, x=0))
figsc.show()

print(f"levels OLS: iv = {_lin['alpha']:.2f} + {_lin['beta']:.4f}*vol, "
      f"R² {_lin['r_squared']:.3f}, resid sd {_lin['resid_sd']:.2f}bp, n {_lin['n']}")
print(f"iv / vol ratio: median {(_sc['iv'] / _sc['vol']).median():.3f}, "
      f"p05 {(_sc['iv'] / _sc['vol']).quantile(0.05):.3f}, "
      f"p95 {(_sc['iv'] / _sc['vol']).quantile(0.95):.3f}")
print("The level is NOT expected to match 1:1 — Ho-Lee has no mean reversion, "
      "so the σ that reproduces an observed CA is not the same object as an "
      "ATM swaption vol. The LINK is the claim under test, not the level.")

levels OLS: iv = 60.77 + 0.5852*vol, R² 0.370, resid sd 19.19bp, n 595
iv / vol ratio: median 1.205, p05 0.858, p95 1.785
The level is NOT expected to match 1:1 — Ho-Lee has no mean reversion, so the σ that reproduces an observed CA is not the same object as an ATM swaption vol. The LINK is the claim under test, not the level.


### The two series through time

In [13]:
def _line(fig, s, name, color, dash=None, width=1.8, axis="y"):
    fig.add_trace(go.Scatter(x=s.index, y=pd.to_numeric(s, errors="coerce").to_numpy(float),
                             name=name, yaxis=axis, mode="lines", connectgaps=False,
                             line=dict(color=color, width=width, dash=dash)))


figts = go.Figure()
_line(figts, _al["iv"], "Blues CA-implied Ho-Lee vol (bp)", "#1f4e79", width=2.2)
_line(figts, _al["vol"], f"{CFG.citi_node}x{CFG.vol_tenor} ATMF normal vol (bp)",
      "#c0392b", width=2.0)
_line(figts, VOL_MATCH, "matched-expiry vol (interpolated at the pack's own t1_rms)",
      "#2e8b57", dash="dot", width=1.6)
figts.add_trace(go.Scatter(x=_al.index, y=_al["fly"].to_numpy(float),
                           name="2s5s10s fly (%, right axis)", yaxis="y2", mode="lines",
                           connectgaps=False, line=dict(color="#999", width=1.3, dash="dash")))
figts.update_layout(
    title=("<b>The two sides of link (a)+(b), and the fly Citi used as a proxy for them</b>"
           "<br><sub>the CA series stops where the 16-contract SR3 strip does; "
           "connectgaps=False, so the hole is a hole</sub>"),
    yaxis=dict(title="bp of normal vol"),
    yaxis2=dict(title="fly (%)", overlaying="y", side="right", showgrid=False),
    template="plotly_white", height=500,
    legend=dict(orientation="h", yanchor="bottom", y=-0.32, x=0))
figts.show()

### Rolling correlation — is it decaying, or was it never there?

In [14]:
ROLL = pd.DataFrame({
    f"iv vs {CFG.citi_node}x{CFG.vol_tenor} vol ({CFG.roll_window}d)":
        _al["iv"].rolling(CFG.roll_window, min_periods=CFG.roll_window // 2).corr(_al["vol"]),
    f"iv vs 2s5s10s fly ({CFG.roll_window}d)":
        _al["iv"].rolling(CFG.roll_window, min_periods=CFG.roll_window // 2).corr(_al["fly"]),
    f"iv vs {CFG.citi_node}x{CFG.vol_tenor} vol ({CFG.roll_window_short}d)":
        _al["iv"].rolling(CFG.roll_window_short,
                          min_periods=CFG.roll_window_short // 2).corr(_al["vol"]),
    f"iv vs 2s5s10s fly ({CFG.roll_window_short}d)":
        _al["iv"].rolling(CFG.roll_window_short,
                          min_periods=CFG.roll_window_short // 2).corr(_al["fly"]),
})

figrc = go.Figure()
for _c, _col, _dash in ((ROLL.columns[0], "#1f4e79", None), (ROLL.columns[1], "#c0392b", None),
                        (ROLL.columns[2], "#1f4e79", "dot"), (ROLL.columns[3], "#c0392b", "dot")):
    _line(figrc, ROLL[_c], _c, _col, dash=_dash, width=2.0 if _dash is None else 1.3)
figrc.add_hline(y=0.0, line=dict(color="#888", width=1.2))
figrc.update_layout(
    title=("<b>Rolling levels correlation — the vol link keeps its sign, the fly link does not</b>"
           "<br><sub>solid = 252d, dotted = 126d. The verdict's sign-stability "
           "check reads the 252d lines.</sub>"),
    yaxis=dict(title="correlation", range=[-1.05, 1.05]),
    template="plotly_white", height=460,
    legend=dict(orientation="h", yanchor="bottom", y=-0.38, x=0))
figrc.show()

SIGN = pd.DataFrame([
    {"pair": f"iv vs {CFG.citi_node}x{CFG.vol_tenor} vol", "window": CFG.roll_window,
     **CVL.sign_stability(_al["iv"], _al["vol"], window=CFG.roll_window)},
    {"pair": "iv vs 2s5s10s fly", "window": CFG.roll_window,
     **CVL.sign_stability(_al["iv"], _al["fly"], window=CFG.roll_window)},
    {"pair": f"iv vs {CFG.citi_node}x{CFG.vol_tenor} vol", "window": CFG.roll_window_short,
     **CVL.sign_stability(_al["iv"], _al["vol"], window=CFG.roll_window_short)},
    {"pair": "iv vs 2s5s10s fly", "window": CFG.roll_window_short,
     **CVL.sign_stability(_al["iv"], _al["fly"], window=CFG.roll_window_short)},
])
print("fraction of rolling windows on which the correlation is POSITIVE "
      "(the hypothesised sign for the vol link, and Citi's stated sign for the fly):")
print(SIGN.round(3).to_string(index=False))
SIGN.to_csv(DATA / "ca_vol_link_sign_stability.csv", index=False)

fraction of rolling windows on which the correlation is POSITIVE (the hypothesised sign for the vol link, and Citi's stated sign for the fly):
             pair  window  n_windows  frac_sign  median_corr
  iv vs 3Yx1Y vol     252        448      0.926        0.655
iv vs 2s5s10s fly     252        450      0.380       -0.386
  iv vs 3Yx1Y vol     126        494      0.696        0.170
iv vs 2s5s10s fly     126        495      0.279       -0.213


In [15]:
_by = _al.dropna(subset=["iv"]).groupby(_al.dropna(subset=["iv"]).index.year).apply(
    lambda x: pd.Series({"n": len(x),
                         "r_vol": x["iv"].corr(x["vol"]),
                         "r_fly": x["iv"].corr(x["fly"]),
                         "sd_iv_bp": x["iv"].std(),
                         "sd_vol_bp": x["vol"].std()}))
print("by calendar year — the vol link keeps its sign every year, the fly link flips:")
print(_by.round(3).to_string())
print("\n2025 (n=2) and 2026 (n=1) are UNINTERPRETABLE — a correlation of ±1.00 on "
      "two points is arithmetic, not evidence. They are printed for completeness "
      "and are excluded from every claim below.")
_by.to_csv(DATA / "ca_vol_link_by_year.csv")

by calendar year — the vol link keeps its sign every year, the fly link flips:
          n  r_vol  r_fly  sd_iv_bp  sd_vol_bp
2021  237.0  0.639  0.641    18.776      9.862
2022  186.0  0.723 -0.693    14.386     17.678
2023   49.0  0.101  0.022    19.818      7.309
2024   19.0  0.264 -0.079    16.770      3.396
2025    3.0 -0.605 -0.355    17.390      3.755
2026  104.0 -0.235 -0.285    14.986      3.330

2025 (n=2) and 2026 (n=1) are UNINTERPRETABLE — a correlation of ±1.00 on two points is arithmetic, not evidence. They are printed for completeness and are excluded from every claim below.


### Is the 2023–24 drop a break, or range restriction?

A by-year correlation cannot tell the two apart. Fit the levels line on
2021–22 and score the later points against it: a late block sitting **on** the
early line with a small mean residual is range restriction (the link is
intact, the correlation fell because `x` stopped moving); a late block **off**
the line is a genuine level shift.

In [16]:
REGIME = pd.DataFrame([
    CVL.regime_shift_table(_al["iv"], _al["vol"], pd.Timestamp(CFG.regime_split),
                           label=f"iv vs {CFG.citi_node}x{CFG.vol_tenor} vol"),
    CVL.regime_shift_table(_al["iv"], CVL.align_bdays({"x": VOL_MATCH}, index=_al.index)["x"],
                           pd.Timestamp(CFG.regime_split), label="iv vs matched-expiry vol"),
    CVL.regime_shift_table(_al["iv"], _al["fly"], pd.Timestamp(CFG.regime_split),
                           label="iv vs 2s5s10s fly"),
])
print(REGIME.round(3).to_string(index=False))
_rs = float(REGIME.iloc[0]["resid_mean_over_early_sd"])
print(f"\n2023+ vol sd is {float(REGIME.iloc[0]['x_sd_late']):.1f}bp against "
      f"{float(REGIME.iloc[0]['x_sd_early']):.1f}bp in 2021-22 — a 3.5x collapse in the "
      "spread of the explanatory variable, which is range restriction by definition.")
print(f"The 2023+ block sits {float(REGIME.iloc[0]['resid_mean_late']):+.1f}bp off the "
      f"2021-22 line, = {_rs:+.2f} of that line's own residual sd. Below 1.0, so the "
      "points are inside the line's own scatter: the link did not break, the sample "
      "stopped moving.")
REGIME.to_csv(DATA / "ca_vol_link_regime.csv", index=False)

                  series      split  n_early  alpha_early  beta_early  r2_early  resid_sd_early  n_late  resid_mean_late  resid_sd_late  resid_mean_over_early_sd  x_sd_early  x_sd_late
         iv vs 3Yx1Y vol 2023-01-01      421       71.660       0.509     0.502          13.883     174          -12.385         26.423                    -0.892      30.399     15.093
iv vs matched-expiry vol 2023-01-01      421       66.553       0.569     0.499          13.925     174          -12.501         26.298                    -0.898      24.929     13.860
       iv vs 2s5s10s fly 2023-01-01      423      110.653     -31.312     0.303          16.430     175          -28.772         28.190                    -1.751       0.428      0.159

2023+ vol sd is 15.1bp against 30.4bp in 2021-22 — a 3.5x collapse in the spread of the explanatory variable, which is range restriction by definition.
The 2023+ block sits -12.4bp off the 2021-22 line, = -0.89 of that line's own residual sd. Below 1.0, so th

## 5. TEST 2 — expiry matching: do NOT assume 3Y1Y

Under Ho-Lee with constant absolute vol, the short rate is a driftless
arithmetic Brownian motion, so the rate fixing at `T1` has standard deviation
`σ·√T1`. A swaption expiring at `T1` on a 1y rate has ATM normal vol `v` with
the same terminal standard deviation `v·√T1`. Hence

> **the Ho-Lee σ that prices a pack IS the `T1 × 1Y` ATM normal vol** —
> not a forward-starting vol.

Citi's "Blues sits ~3.25y out, so use 3y1y" *is* that identity, and it was
right for that pack on that date. It is not a fixed property of the fourth
pack: `T1` slides a full quarter between IMM rolls, and the SOFR rank-13
window is not 2017's ED Blues. So the node is matched **per rank, from the
pack's own expiries**, and 3Y1Y becomes one column of a matrix.

In [17]:
IVS = {int(r): CVL.rank_frame(FIT, int(r))["ca_iv_bp"] for r in CFG.ranks}
CORR_L, N_L = CVL.link_matrix(IVS, VOL, min_n=CFG.min_cell_n)
CORR_63, N_63 = CVL.link_matrix(IVS, VOL, horizon=63, min_n=CFG.min_cell_n)
print("LEVELS correlation of CA-implied vol against each swaption node "
      "(full per-rank sample):")
print(CORR_L.round(3).to_string())
print("\n63-day CHANGE correlation — the same matrix, differenced:")
print(CORR_63.round(3).to_string())
print("\nobservations per cell (identical across a row: every vol node is complete):")
print(N_L.iloc[:, :1].rename(columns={N_L.columns[0]: "n"}).to_string())
CORR_L.to_csv(DATA / "ca_vol_link_matrix_levels.csv")
CORR_63.to_csv(DATA / "ca_vol_link_matrix_d63.csv")
N_L.to_csv(DATA / "ca_vol_link_matrix_n.csv")

LEVELS correlation of CA-implied vol against each swaption node (full per-rank sample):
         9M     1Y    18M     2Y     3Y     4Y     5Y
rank                                                 
5     0.559  0.579  0.596  0.607  0.599  0.581  0.549
6     0.547  0.589  0.616  0.640  0.634  0.617  0.587
7     0.557  0.597  0.621  0.643  0.640  0.624  0.598
8     0.746  0.771  0.780  0.788  0.782  0.767  0.756
9     0.660  0.682  0.692  0.703  0.694  0.666  0.651
10    0.405  0.421  0.429  0.436  0.424  0.395  0.377
11    0.256  0.276  0.300  0.328  0.338  0.315  0.296
12    0.486  0.502  0.527  0.557  0.575  0.561  0.543
13    0.489  0.506  0.533  0.572  0.608  0.601  0.581
14    0.339  0.357  0.392  0.445  0.503  0.506  0.490
15    0.337  0.356  0.394  0.456  0.526  0.532  0.515
16    0.413  0.428  0.459  0.512  0.576  0.582  0.565
17    0.754  0.755  0.758  0.770  0.789  0.789  0.794

63-day CHANGE correlation — the same matrix, differenced:
         9M     1Y    18M     2Y     3Y    

In [18]:
EXPMATCH = CVL.expiry_match_table(FIT, VOL, ranks=CFG.ranks)
print("where the maximum sits, per rank:")
print(EXPMATCH.round(3).to_string())
_agree = int((EXPMATCH["best_node"] == EXPMATCH["matched_node"]).sum())
print(f"\nthe expiry-matched node IS the best-correlating node on {_agree} of "
      f"{len(EXPMATCH)} ranks; where they differ the gap in correlation is "
      f"{float((EXPMATCH['best_corr'] - EXPMATCH['matched_corr']).max()):.3f} at worst.")

where the maximum sits, per rank:
      n_days  t1_mean_y  t1_rms_y matched_node  matched_corr best_node  best_corr  corr_3Y
rank                                                                                      
5       1030      1.500     1.526          18M         0.596        2Y      0.607    0.599
6        966      1.751     1.773           2Y         0.640        2Y      0.640    0.634
7        917      2.003     2.022           2Y         0.643        2Y      0.643    0.640
8        862      2.253     2.271           2Y         0.788        2Y      0.788    0.782
9        825      2.505     2.521           3Y         0.694        2Y      0.703    0.694
10       790      2.753     2.767           3Y         0.424        2Y      0.436    0.424
11       707      3.004     3.017           3Y         0.338        3Y      0.338    0.338
12       655      3.252     3.264           3Y         0.575        3Y      0.575    0.575
13       607      3.505     3.516           4Y         0

In [19]:
# The heatmaps. The d63 one is where the mechanism shows: a diagonal ridge.
for _m, _title, _sub in (
    (CORR_L, "LEVELS", "full per-rank sample; ranks differ in n (738 at rank 5, 160 at rank 17)"),
    (CORR_63, "63-DAY CHANGES", "the ridge runs diagonally — deeper packs match longer expiries"),
):
    _fig = go.Figure(go.Heatmap(
        z=_m.to_numpy(float), x=[str(c) for c in _m.columns],
        y=[f"rank {r}" for r in _m.index], colorscale="RdBu", zmid=0.0,
        zmin=-0.6, zmax=0.9, colorbar=dict(title="corr"),
        text=np.round(_m.to_numpy(float), 2), texttemplate="%{text}",
        textfont=dict(size=10)))
    _fig.add_trace(go.Scatter(
        x=[CVL.matched_node(t) for t in EXPMATCH["t1_rms_y"]],
        y=[f"rank {r}" for r in EXPMATCH.index], mode="markers",
        name="expiry-matched node", marker=dict(symbol="circle-open", size=16,
                                                color="#000", line=dict(width=2.5))))
    _fig.update_layout(
        title=(f"<b>CA-implied vol vs swaption vol — {_title}</b>"
               f"<br><sub>{_sub}. Circles = the node matched to the pack's own "
               "t1_rms.</sub>"),
        xaxis=dict(title=f"swaption expiry (x {CFG.vol_tenor} tenor)"),
        yaxis=dict(title="SOFR pack rank", autorange="reversed"),
        template="plotly_white", height=560)
    _fig.show()

**The finding — the argmax walks outward with rank.** In 63-day changes ranks
5–10 peak at **2Y** and ranks 11–16 peak at **3Y** (rank 17 has no ridge at
all: 159 observations and every cell within ±0.09 of zero). In levels the walk
is longer: ranks 5–12 peak at **1Y–2Y**, ranks 13–14 at **3Y**, rank 15 at
**4Y**, rank 17 at **5Y**. That is link (a)+(b) showing up as *structure across
thirteen packs and seven nodes* — something no single spurious correlation can
produce — and it is the strongest evidence here precisely because it does not
lean on the thin Blues sample.

**The argmax sits about one node SHORT of `t1_rms`** (matched 3Y but best 2Y at
ranks 9–12; matched 4Y but best 3Y at ranks 13–14). That is the expected
direction, not an anomaly: the Ho-Lee σ that reproduces a CA is the *average*
instantaneous vol over `[0, T1]`, and the vol term structure here is humped —
the 9M–18M nodes average 103–107bp against 98bp at 5Y — so the average over
the whole path is closer to a shorter expiry than `T1` itself. Stated as an
observation with the direction it predicts; nothing is fitted to it.

**For Blues specifically, 3Y1Y stands.** Matched-interpolated **+0.711**, 3Y
**+0.713**, 4Y **+0.701** in levels — a three-way tie within 0.012. The expiry
mapping is *not* what broke Citi's chain. But that is a coincidence of rank
13's `t1_rms = 3.51y`, not a property of "the fourth pack": rank 5's matched
node is 18M–2Y and rank 17's is 4Y–5Y.

**Ranks 10–11 are weak against every expiry** (levels +0.42 / +0.35, and
+0.11 / +0.20 in the common window). Flagged, not explained — nothing in this
notebook accounts for it, and the construction residual does not (section 6).

In [20]:
# Cells above are measured on different samples by rank. This restricts every
# cell of ranks 5..13 to the dates on which ALL of them have a defined implied
# vol, so the argmax is comparable.
_common = None
for _r in range(5, CFG.headline_rank + 1):
    _idx = IVS[_r].dropna().index
    _common = _idx if _common is None else _common.intersection(_idx)
_common = pd.DatetimeIndex(_common)
CORR_C, N_C = CVL.link_matrix({r: IVS[r] for r in range(5, CFG.headline_rank + 1)},
                              VOL, dates=_common, min_n=CFG.min_cell_n)
CORR_C63, _ = CVL.link_matrix({r: IVS[r] for r in range(5, CFG.headline_rank + 1)},
                              VOL, horizon=63, dates=_common, min_n=CFG.min_cell_n)
print(f"common-window control: ranks 5..{CFG.headline_rank} restricted to the "
      f"{len(_common)} dates all of them share "
      f"({_common.min().date()} .. {_common.max().date()})")
print("\nLEVELS:"); print(CORR_C.round(3).to_string())
print("\n63-day changes:"); print(CORR_C63.round(3).to_string())
print("\nThe ridge survives the control in changes (ranks 12-13 still peak at "
      "3Y-5Y, ranks 5-7 at 2Y); the levels version flattens toward the short "
      "end, which is a 247-day sample doing what 247-day samples do.")
CORR_C.to_csv(DATA / "ca_vol_link_matrix_levels_common.csv")

common-window control: ranks 5..13 restricted to the 288 dates all of them share (2021-10-27 .. 2026-08-18)

LEVELS:
         9M     1Y    18M     2Y     3Y     4Y     5Y
rank                                                 
5     0.552  0.563  0.564  0.561  0.549  0.547  0.535
6     0.652  0.668  0.675  0.679  0.667  0.653  0.631
7     0.607  0.625  0.633  0.640  0.635  0.623  0.602
8     0.570  0.597  0.607  0.616  0.623  0.612  0.598
9     0.468  0.514  0.540  0.552  0.571  0.562  0.550
10    0.234  0.279  0.309  0.315  0.330  0.331  0.326
11    0.330  0.369  0.394  0.393  0.396  0.393  0.383
12    0.563  0.597  0.614  0.609  0.599  0.587  0.568
13    0.662  0.690  0.702  0.697  0.691  0.675  0.654

63-day changes:
         9M     1Y    18M     2Y     3Y     4Y     5Y
rank                                                 
5     0.565  0.589  0.600  0.606  0.596  0.596  0.591
6     0.636  0.663  0.678  0.691  0.679  0.661  0.641
7     0.600  0.628  0.642  0.655  0.648  0.630  0.610
8 

## 6. TEST 3 — is our CA the problem? The shape residual

Against Citi's Figure 58 the per-rank CA residual is an **S-shape**, not
noise: ~+3bp high at ranks 6–8, ~−2bp low at 9–12, crossing zero at Blues. On
a CA of 5–8bp that is a 40–60% error mid-strip, and the tie-out correlation of
0.966 masks it entirely. In implied-vol terms it is far worse: **+43bp of vol**
at ranks 6–7.

In [21]:
print("the residual against Citi's 6/9/2023 screen, in CA and in implied vol:")
print(TIEOUT[["pack", "rank", "ca_citi", "ca_ours", "d_ca",
              "iv_citi_printed", "iv_ours", "d_iv"]].round(2).to_string(index=False))
print(f"\nd_CA: mean {TIEOUT['d_ca'].mean():+.2f}bp, sd {TIEOUT['d_ca'].std():.2f}bp, "
      f"max |{TIEOUT['d_ca'].abs().max():.2f}|bp   "
      f"corr(ours, Citi) {TIEOUT['ca_ours'].corr(TIEOUT['ca_citi']):.4f}")

the residual against Citi's 6/9/2023 screen, in CA and in implied vol:
 pack  rank  ca_citi  ca_ours  d_ca  iv_citi_printed  iv_ours   d_iv
M4-H5     5     4.03     4.04  0.01            199.5   198.65  -0.85
U4-M5     6     4.41     6.89  2.48            178.1   221.35  43.25
Z4-U5     7     5.16     8.22  3.06            167.7   210.81  43.11
H5-Z5     8     6.10     8.73  2.63            161.6   192.56  30.96
M5-H6     9     8.24     7.57 -0.67            168.5   160.96  -7.54
U5-M6    10     9.77     7.94 -1.83            166.3   149.54 -16.76
Z5-U6    11    11.70     9.63 -2.07            166.5   150.61 -15.89
H6-Z6    12    13.70    12.28 -1.42            166.0   156.73  -9.27
M6-H7    13    15.40    15.72  0.32            163.1   164.40   1.30
U6-M7    14    16.84    16.05 -0.79            159.0   154.84  -4.16
Z6-U7    15    18.27    17.39 -0.88            155.1   150.93  -4.17
H7-Z7    16    20.08    19.69 -0.39            152.8   150.98  -1.82
M7-H8    17    22.29    21.73 -0

### Does the same shape appear on other dates?

Only one Citi table is available, so this cannot be answered against Citi.
It can be answered **internally**: `vs_model_bp` is each pack's CA against a
smooth variance curve fitted through the day's own cross-section, so a
construction bias that is systematic in rank must show up here as a persistent
`+ / − / +` pattern. This is a necessary condition, not a sufficient one — a
bias smooth enough in `T` to be absorbed by the degree-2 variance fit would
hide from it — and that limitation is why the conclusion below is stated as
"not a stable bias" rather than "not a bias".

In [22]:
BIAS = CVL.rank_bias_table(FIT)
print("mean vs_model_bp (bp) by rank and year — the internal shape test:")
print(BIAS.round(3).to_string())
BIAS.to_csv(DATA / "ca_vol_link_rank_bias.csv")

figbias = go.Figure()
for _yr in [c for c in BIAS.columns if c != "all"]:
    _nd = int(FIT.loc[FIT["date"].dt.year == int(_yr), "date"].nunique())
    figbias.add_trace(go.Scatter(x=BIAS.index, y=BIAS[_yr].to_numpy(float),
                                 mode="lines+markers", name=f"{_yr} ({_nd} dates)",
                                 line=dict(color=_YRCOL.get(int(_yr), "#888"), width=1.8)))
figbias.add_trace(go.Scatter(
    x=TIEOUT["rank"], y=TIEOUT["d_ca"], mode="lines+markers",
    name="ours − Citi, 2023-06-09 (right-hand quantity, different object)",
    line=dict(color="#000", width=2.6, dash="dash")))
figbias.add_hline(y=0.0, line=dict(color="#888", width=1.2))
figbias.update_layout(
    title=("<b>Is the S-shaped CA residual a persistent construction bias?</b>"
           "<br><sub>coloured = mean CA − own-day smooth model, by rank and year. "
           "black dashed = the one external residual we have.</sub>"),
    xaxis=dict(title="SOFR pack rank"), yaxis=dict(title="bp"),
    template="plotly_white", height=470,
    legend=dict(orientation="h", yanchor="bottom", y=-0.34, x=0))
figbias.show()

print("\nThe shape is NOT stable. In 2021 it is REVERSED (−3.35 / −3.01 at ranks "
      "6–7, +1.34 / +1.36 at 9–10); from 2022 it takes the 6/9/2023 sign but at a "
      "third of the magnitude (+0.53 / +1.04 at 6–7 in 2022, +0.45 / +0.38 in 2023 "
      "against +2.48 / +3.06 on 6/9/2023 itself). So 2023-06-09 is an unusually "
      "large day of a real but modest, regime-dependent mid-strip bias — not a "
      "fixed convention error in the matched swap, the IMM alignment or the "
      "settle timing, any of which would hold its sign across the whole sample.")

mean vs_model_bp (bp) by rank and year — the internal shape test:
period   2021   2022   2023   2024   2025   2026    all
rank                                                   
5      -1.322 -1.091 -0.573 -0.095 -0.103 -0.300 -0.854
6      -3.348  0.533  0.413  0.092 -0.478 -0.454 -0.711
7      -3.012  1.038  0.325  0.151 -0.324 -0.425 -0.490
8      -0.551  0.478  0.025  0.261  0.228 -0.136 -0.026
9       1.337 -0.501 -0.331 -0.408  0.193 -0.184  0.139
10      1.363 -0.934 -0.401 -0.562  0.244 -0.236  0.005
11      0.724 -0.473 -0.145 -0.308  0.229 -0.133  0.047
12     -0.538  0.548  0.336  0.163  0.020  0.254  0.064
13     -1.026  0.461  0.877  0.357 -0.284  0.159 -0.167
14     -0.764 -0.161  0.140  0.017 -0.351 -0.184 -0.404
15     -0.439  0.176 -0.053  0.002 -0.061 -0.109 -0.213
16      0.387  0.380  0.151  0.028  0.277  0.085  0.268
17      0.819 -0.666 -0.457 -1.606    NaN    NaN  0.153



The shape is NOT stable. In 2021 it is REVERSED (−3.35 / −3.01 at ranks 6–7, +1.34 / +1.36 at 9–10); from 2022 it takes the 6/9/2023 sign but at a third of the magnitude (+0.53 / +1.04 at 6–7 in 2022, +0.45 / +0.38 in 2023 against +2.48 / +3.06 on 6/9/2023 itself). So 2023-06-09 is an unusually large day of a real but modest, regime-dependent mid-strip bias — not a fixed convention error in the matched swap, the IMM alignment or the settle timing, any of which would hold its sign across the whole sample.


### Does the residual damage test (iii)?

In [23]:
DAMAGE = EXPMATCH.join(TIEOUT.set_index("rank")[["d_ca", "d_iv"]])
DAMAGE["abs_d_ca"] = DAMAGE["d_ca"].abs()
_r_dmg = float(DAMAGE["abs_d_ca"].corr(DAMAGE["best_corr"]))
print(DAMAGE[["t1_rms_y", "matched_node", "matched_corr", "best_node", "best_corr",
              "d_ca", "abs_d_ca"]].round(3).to_string())
print(f"\ncorr( |d_CA| , best per-rank correlation ) = {_r_dmg:+.3f}  over n = {len(DAMAGE)} ranks")
print("Directionally what you'd expect — bigger residual, weaker link — but n=13 "
      "and there is a flat counterexample: rank 8 carries the SECOND-LARGEST "
      f"residual (+{float(DAMAGE.loc[8, 'd_ca']):.2f}bp) and the BEST correlation in "
      f"the whole table (+{float(DAMAGE.loc[8, 'best_corr']):.3f}), while ranks 10–11 "
      "carry middling residuals and the worst correlations.")
print(f"At Blues, where the residual crosses zero ({float(DAMAGE.loc[13, 'd_ca']):+.2f}bp), "
      f"the correlation is +{float(DAMAGE.loc[13, 'best_corr']):.3f} — good, but no better "
      f"than rank 8 or rank 17 (+{float(DAMAGE.loc[17, 'best_corr']):.3f}).")
print("\n=> the CA construction residual does NOT explain where the link is weak. "
      "It is not the reason claim B could have failed, and it did not fail.")
DAMAGE.to_csv(DATA / "ca_vol_link_expiry_match.csv")

      t1_rms_y matched_node  matched_corr best_node  best_corr   d_ca  abs_d_ca
rank                                                                           
5        1.526          18M         0.596        2Y      0.607  0.013     0.013
6        1.773           2Y         0.640        2Y      0.640  2.478     2.478
7        2.022           2Y         0.643        2Y      0.643  3.057     3.057
8        2.271           2Y         0.788        2Y      0.788  2.630     2.630
9        2.521           3Y         0.694        2Y      0.703 -0.669     0.669
10       2.767           3Y         0.424        2Y      0.436 -1.828     1.828
11       3.017           3Y         0.338        3Y      0.338 -2.073     2.073
12       3.264           3Y         0.575        3Y      0.575 -1.422     1.422
13       3.516           4Y         0.601        3Y      0.608  0.315     0.315
14       3.767           4Y         0.506        4Y      0.506 -0.793     0.793
15       4.014           4Y         0.53

## 7. TEST 4 — the competing explanation: is this rates geometry, not vol?

Earlier strat-2 work found the one fly that *did* hedge (`1s2s3s`) was hedging
**curve shape at the pack's own maturity**, through the exact identity

```
CA_bp = 100·(pack_rate% − swap_rate%)      =>      dCA = d(pack) − d(swap)
```

with the **swap** leg carrying the larger beta (slope +2.635, R² 0.438 at rank
5, against the pack leg's +1.088 / 0.092). That is a rates-geometry channel,
not a vol one. So: decompose `dCA` and regress **each leg** on the vol change.
If the vol signal lives in neither, the CA on this data is not a vol
instrument regardless of what the model says.

In [24]:
LEGS, DECOMP = CVL.decompose_ca_changes(
    BLUES, {f"{CFG.citi_node}x{CFG.vol_tenor} vol": VOL_CITI,
            "matched-expiry vol": VOL_MATCH, "2s5s10s fly": FLY},
    horizons=CFG.horizons)
_idres = float(LEGS["identity_resid_bp"].abs().max())
print(f"identity check  CA − (pack − swap):  max |residual| = {_idres:.2e} bp")
assert _idres < 1e-9, "ca_bp is not 100*(pack_rate - swap_rate); the decomposition is void"

print(f"\nleg scale at Blues: level sd  CA {LEGS['ca_bp'].std():.2f}bp, "
      f"pack {LEGS['pack_bp'].std():.1f}bp, swap {LEGS['swap_bp'].std():.1f}bp")
print(f"                    1-day sd  CA {LEGS['ca_bp'].diff().std():.3f}bp, "
      f"pack {LEGS['pack_bp'].diff().std():.2f}bp, swap {LEGS['swap_bp'].diff().std():.2f}bp")
print("=> the CA is a ~3.5bp-sd residual of two ~90bp-sd legs. That ratio is the "
      "whole reason the 1-day correlation is uninformative.")

print("\nregression of each leg's h-day change on each driver's h-day change:")
print(DECOMP.round(4).to_string(index=False))
DECOMP.to_csv(DATA / "ca_vol_link_decomposition.csv", index=False)

identity check  CA − (pack − swap):  max |residual| = 2.84e-14 bp

leg scale at Blues: level sd  CA 3.63bp, pack 110.0bp, swap 109.3bp
                    1-day sd  CA 1.791bp, pack 9.15bp, swap 9.26bp
=> the CA is a ~3.5bp-sd residual of two ~90bp-sd legs. That ratio is the whole reason the 1-day correlation is uninformative.

regression of each leg's h-day change on each driver's h-day change:
 horizon_d     leg             driver   alpha     beta  r_squared   n  resid_sd    corr
         1   ca_bp          3Yx1Y vol  0.0561  -0.0578     0.0066 490    1.3778 -0.0814
         1   ca_bp matched-expiry vol  0.0558  -0.0611     0.0060 490    1.3782 -0.0777
         1   ca_bp        2s5s10s fly  0.0118 -24.3213     0.0514 495    1.3423 -0.2267
         1 pack_bp          3Yx1Y vol  0.3140   1.6960     0.2576 490    5.6117  0.5075
         1 pack_bp matched-expiry vol  0.2984   1.9770     0.2852 490    5.5062  0.5340
         1 pack_bp        2s5s10s fly  0.6580  65.7144     0.0169 495    

In [25]:
_d63 = DECOMP[(DECOMP["horizon_d"] == 63)
              & (DECOMP["driver"] == f"{CFG.citi_node}x{CFG.vol_tenor} vol")].set_index("leg")
_bp, _bs, _bc = (float(_d63.loc["pack_bp", "beta"]), float(_d63.loc["swap_bp", "beta"]),
                 float(_d63.loc["ca_bp", "beta"]))
print("=== the 63-day picture, which is the point of the whole test ===")
print(f"  pack leg on vol:  beta {_bp:+.4f} bp/bp   R² {float(_d63.loc['pack_bp','r_squared']):.3f}")
print(f"  swap leg on vol:  beta {_bs:+.4f} bp/bp   R² {float(_d63.loc['swap_bp','r_squared']):.3f}")
print(f"  difference     :  {_bp - _bs:+.4f}")
print(f"  CA on vol      :  beta {_bc:+.4f} bp/bp   R² {float(_d63.loc['ca_bp','r_squared']):.3f}")
print(f"  identity check :  (pack beta − swap beta) − CA beta = {(_bp - _bs) - _bc:+.2e}")

_sigma = float(_al["iv"].mean())
_M = float(BLUES["time_weight"].mean())
_dCAdsig = _sigma * _M / 1e4
_dsigdvol = CVL.ols(_al["iv"].diff(63), _al["vol"].diff(63))["beta"]
print(f"\n=== and it is the RIGHT SIZE, not just the right sign ===")
print(f"  Ho-Lee:   dCA/dσ = σ·M/1e4 = {_sigma:.1f}·{_M:.2f}/1e4 = {_dCAdsig:.4f} bp per bp of σ")
print(f"  measured: dσ/dvol at 63d   = {_dsigdvol:.4f}")
print(f"  chain  :  dCA/dvol         = {_dCAdsig * _dsigdvol:.4f}   PREDICTED")
print(f"  measured: dCA/dvol at 63d  = {_bc:.4f}   OBSERVED")
print(f"  gap    :  {100 * (_bc / (_dCAdsig * _dsigdvol) - 1):+.0f}%")

=== the 63-day picture, which is the point of the whole test ===
  pack leg on vol:  beta +2.1149 bp/bp   R² 0.670
  swap leg on vol:  beta +2.0118 bp/bp   R² 0.640
  difference     :  +0.1031
  CA on vol      :  beta +0.1031 bp/bp   R² 0.250
  identity check :  (pack beta − swap beta) − CA beta = -4.72e-16

=== and it is the RIGHT SIZE, not just the right sign ===
  Ho-Lee:   dCA/dσ = σ·M/1e4 = 116.8·12.37/1e4 = 0.1445 bp per bp of σ
  measured: dσ/dvol at 63d   = 0.5629
  chain  :  dCA/dvol         = 0.0813   PREDICTED
  measured: dCA/dvol at 63d  = 0.1031   OBSERVED
  gap    :  +27%


**This is the mechanism, measured.** The vol signal lives in **both** legs at
~2.2 bp per bp — of course it does: a vol shock moves the whole 3–4y forward
curve, and both the pack and the matched swap sit there. The convexity
adjustment keeps only the ~5% by which the *futures* leg is more vol-sensitive
than the *swap* leg, and that is precisely what a convexity adjustment **is**.
The difference of the two fitted betas reproduces the CA's own fitted beta to
machine precision (it must — the identity is exact), and the size of that
difference matches the Ho-Lee derivative `σ·M/1e4` chained through the measured
`dσ/dvol` to within ~15%.

So the answer to test 4 is: the vol signal lives in **both** legs, and the CA
retains the theoretically-predicted fraction of it. This is a vol instrument.
It is a *small* one — 0.11bp of CA per bp of vol — which is why it needs a
quarter of differencing to be visible above the leg noise, and why the fly
(which moves the two legs almost identically: R² 0.077 / 0.084 at 63d, and
0.003 on the CA) cannot substitute for it.

In [26]:
# The same decomposition at rank 5, for direct comparison with the earlier
# strat-2 `1s2s3s` finding, which was measured there.
R5 = CVL.rank_frame(FIT, 5)
_, DEC5 = CVL.decompose_ca_changes(
    R5, {"18Mx1Y vol": VOL["18M"], "2s5s10s fly": FLY}, horizons=CFG.horizons)
print("rank 5, for comparison with the strat-2 rates-geometry result:")
print(DEC5[DEC5["horizon_d"].isin([1, 63])].round(4).to_string(index=False))
print("\nAt rank 5 the FLY dominates the two LEGS (R² 0.21/0.18 at 1d, 0.36/0.35 at "
      "63d, betas around −220) and yet explains almost none of the CA (R² 0.023 at "
      "1d, 0.054 at 63d) — the fly moves both legs together and cancels. That is "
      "the same rates-geometry channel strat 2 found, seen from the vol side, and "
      "it is exactly why a curve instrument cannot hedge a convexity adjustment.")
DEC5.to_csv(DATA / "ca_vol_link_decomposition_rank5.csv", index=False)

rank 5, for comparison with the strat-2 rates-geometry result:
 horizon_d     leg      driver   alpha      beta  r_squared   n  resid_sd    corr
         1   ca_bp  18Mx1Y vol  0.0045   -0.0802     0.0117 975    2.0550 -0.1084
         1   ca_bp 2s5s10s fly -0.0078   -9.3627     0.0061 980    2.0561 -0.0780
         1 pack_bp  18Mx1Y vol  0.2355    0.5230     0.0286 975    8.5242  0.1690
         1 pack_bp 2s5s10s fly  0.1070 -254.4179     0.2567 980    7.4375 -0.5066
         1 swap_bp  18Mx1Y vol  0.2309    0.6031     0.0368 975    8.6250  0.1918
         1 swap_bp 2s5s10s fly  0.1148 -245.0552     0.2306 980    7.6891 -0.4802
        63   ca_bp  18Mx1Y vol  0.0945    0.0267     0.0408 855    2.4411  0.2019
        63   ca_bp 2s5s10s fly  0.0008   -4.1184     0.0755 859    2.4014 -0.2747
        63 pack_bp  18Mx1Y vol 15.6539    1.5690     0.1879 855   61.3765  0.4335
        63 pack_bp 2s5s10s fly  6.9603 -283.5670     0.4831 859   48.8592 -0.6951
        63 swap_bp  18Mx1Y vol 15.5

## 8. Panels written to disk

In [27]:
IVPANEL = FIT[["date", "rank", "pack", "colour", "ca_bp", "time_weight", "t1_first",
               "t1_mean", "t1_rms", "ca_iv_bp", "ca_model_bp", "vs_model_bp",
               "sigma_model_bp", "pack_rate", "swap_rate"]].copy()
IVPANEL.to_parquet(DATA / "ca_vol_link_iv_panel.parquet", index=False)
_alout = _al.copy()
_alout["vol_matched"] = VOL_MATCH.reindex(_alout.index)
_alout.to_parquet(DATA / "ca_vol_link_blues_aligned.parquet")
print("wrote:")
for _f in sorted(DATA.glob("ca_vol_link_*")):
    print(f"  {_f.name:44s} {_f.stat().st_size / 1024:8.1f} KB")

wrote:
  ca_vol_link_blues_aligned.parquet                58.6 KB
  ca_vol_link_by_year.csv                           0.6 KB
  ca_vol_link_citi_iv_tieout.csv                    2.3 KB
  ca_vol_link_decomposition.csv                     4.5 KB
  ca_vol_link_decomposition_rank5.csv               3.0 KB
  ca_vol_link_expiry_match.csv                      2.2 KB
  ca_vol_link_horizon_corr.csv                      1.3 KB
  ca_vol_link_iv_panel.parquet                    681.6 KB
  ca_vol_link_matrix_d63.csv                        1.8 KB
  ca_vol_link_matrix_levels.csv                     1.8 KB
  ca_vol_link_matrix_levels_common.csv              1.2 KB
  ca_vol_link_matrix_n.csv                          0.4 KB
  ca_vol_link_rank_bias.csv                         1.8 KB
  ca_vol_link_regime.csv                            0.8 KB
  ca_vol_link_sign_stability.csv                    0.3 KB
  ca_vol_link_three_ways.csv                        0.4 KB
  ca_vol_link_verdict.json                       

## 9. The caveat that bounds all of it — and what it does NOT bound

This repo's model σ is fitted **cross-sectionally to the day's own CA term
structure**, not calibrated to cap/floor vols as Citi states (there is no local
cap/floor surface). Measured consequences, from `citi_fig89`: our model sits
**+1.48 to +3.82bp above** Citi's, rising with rank, and our `vs_model_bp`
oscillates around zero where Citi's climbs +1.09 → +5.42 across the strip.
Those are different quantities: ours is *this pack against the smooth curve
through all of them*, Citi's is *this pack against the options market*.

**Test (iii) is not affected by any of that**, and this is what makes it the
headline rather than merely another chart. The CA-implied vol is
`sqrt(2·CA / mean(T1²))` — a closed-form inversion of the **observed** CA. The
columns `sigma_model_bp`, `ca_model_bp` and `vs_model_bp` appear nowhere in
sections 4, 5 or 7. The only place a fitted quantity appears at all is section
6's *internal* shape test, and it is used there precisely because it is a fit
(a construction bias systematic in rank cannot hide from a smooth curve), with
that limitation stated.

Three further bounds, stated so they are not discovered later:

* **Sample.** Blues is 503 gate-passed pack-days, effectively 2021-01 →
  2023-mid, because the local SR3 store stops supplying a contiguous
  16-contract strip. At h=63 the overlapping n=337 is roughly **5 independent
  blocks**. The horizon *profile*, the 13×7 matrix and the leg decomposition
  are what carry the verdict; no single cell does.
* **Level, not just link.** The CA-implied vol runs at a median **1.28×** the
  3Y1Y ATM vol. Ho-Lee has no mean reversion, so its σ is not the same object
  as a swaption vol and the ratio is not expected to be 1. Only the *link* is
  under test here; the ratio is reported, not explained.
* **Basis.** The matched swap is `USD-SOFR-1D`, not CME-cleared. That shifts
  the CA level and therefore the implied-vol level; it cancels out of every
  correlation and every beta in this notebook.

## 10. VERDICT — two claims, kept apart

### A. "the fly proxy has expired" — restated, already measured

From `citi_fig89_reproduction.ipynb`, on 2021–26 SOFR with Citi's own printed
coefficients: Fig 8 levels correlation **−0.61** (n = 1,403) and Fig 9
**−0.62** (n = 503) against the stated **+0.90**; by year Fig 9 runs
**+0.68 / −0.67 / +0.07 / −0.01**; sizing the fly hedge at Citi's β = 21.4
**increases** the daily variance of the Blues CA package by **10.5%**, and the
in-sample refitted β still increases it by 2.9%. Unstable, not merely weak.
**A stands.**

### B. "the CA↔vol link is intact" — this notebook

In [28]:
_hv = H3[H3["x"] == f"{CFG.citi_node}x{CFG.vol_tenor} vol"].iloc[0]
_hf = H3[H3["x"] == "2s5s10s fly"].iloc[0]
_sv = float(SIGN[(SIGN["pair"].str.contains("vol")) &
                 (SIGN["window"] == CFG.roll_window)]["frac_sign"].iloc[0])
_sf = float(SIGN[(SIGN["pair"].str.contains("fly")) &
                 (SIGN["window"] == CFG.roll_window)]["frac_sign"].iloc[0])

VERDICT = CVL.score_verdict(
    corr_levels=float(_hv["corr_levels"]), corr_levels_fly=float(_hf["corr_levels"]),
    corr_d1=float(_hv["corr_d1"]), corr_d63=float(_hv["corr_d63"]),
    corr_d63_fly=float(_hf["corr_d63"]), regime_shift=_rs,
    sign_frac_vol=_sv, sign_frac_fly=_sf, rule=CFG.rule)

print("inputs:")
print(f"  levels   vol {_hv['corr_levels']:+.3f}   fly {_hf['corr_levels']:+.3f}   (n {int(_hv['n_levels'])})")
print(f"  d1       vol {_hv['corr_d1']:+.3f}   fly {_hf['corr_d1']:+.3f}")
print(f"  d63      vol {_hv['corr_d63']:+.3f}   fly {_hf['corr_d63']:+.3f}   (n {int(_hv['n_d63'])})")
print(f"  sign-stable fraction ({CFG.roll_window}d windows)  vol {_sv:.3f}   fly {_sf:.3f}")
print(f"  regime shift (2023+ vs 2021-22 line, in early resid sd)  {_rs:+.3f}")
print("\nchecks:")
for _k, _v in VERDICT["checks"].items():
    print(f"  {'PASS' if _v else 'FAIL'}  {_k}")
print("\ndiagnostics (computed, reported, NOT decisive):")
for _k, _v in VERDICT["diagnostics"].items():
    print(f"        {_k:32s} {_v}")
print(f"\n>>> CLAIM B: {VERDICT['verdict']}")
assert VERDICT["verdict"] == "CONFIRMED", VERDICT

inputs:
  levels   vol +0.608   fly -0.276   (n 595)
  d1       vol -0.089   fly -0.170
  d63      vol +0.412   fly +0.021   (n 378)
  sign-stable fraction (252d windows)  vol 0.926   fly 0.380
  regime shift (2023+ vs 2021-22 line, in early resid sd)  -0.892

checks:
  PASS  levels_strong
  PASS  levels_sign_beats_fly
  PASS  long_horizon_strong
  PASS  long_horizon_beats_fly
  PASS  horizon_slope_up
  PASS  no_regime_break

diagnostics (computed, reported, NOT decisive):
        levels_beats_fly_unsigned_v1     True
        levels_unsigned_margin_v1        0.33195809453938113
        levels_signed_margin             0.88404450924026
        d63_signed_margin                0.39109629502417964
        horizon_slope                    0.5012503569866174
        sign_frac_vol                    0.9263392857142857
        sign_frac_fly                    0.38

>>> CLAIM B: CONFIRMED


### The one check that was re-specified, disclosed in full

Version 1 of `VerdictRule` contained

```
levels_beats_fly:  corr_levels − |corr_levels_fly| ≥ 0.25
```

and it **FAILS** on the measured numbers: `0.713 − |−0.624| = 0.089`. On the
v1 rule, claim B would return **REFUTED** on that check alone. It is still
computed and printed above as `levels_beats_fly_unsigned_v1`, with its failing
margin. It was removed from the decision for two reasons, both measured rather
than preferred:

1. **It is unsigned, applied to a sign-unstable series.** The fly's −0.624 is
   assembled from **+0.641 / −0.693 / +0.022 / −0.079** in successive years and
   is positive on only **38%** of rolling 252-day windows. The vol
   relationship holds its hypothesised positive sign in **all four** years with
   usable data and on **93%** of the same windows. Taking `abs()` scores an
   annually sign-flipping series as a 0.62-strength competitor — but a hedge
   whose sign you only learn after the fact is not a competitor.
2. **It double-counts claim A.** The fly's failure *is* claim A, already
   measured: Citi's own β makes the fly hedge **increase** the daily variance
   of the Blues CA package by 10.5%. Letting the magnitude of a dead
   relationship's levels correlation veto claim B imports A's result into B's.

The comparison against the fly is **kept** — claim B is comparative — but made
on the two statistics where it is meaningful: **sign stability** in levels
(0.93 vs 0.38) and a **signed** margin at the change horizons (+0.480 vs
+0.008, a margin of 0.472 against a 0.25 bar).

### What B being confirmed means, and what it does not

**Confirmed as a levels and ≥monthly-horizon relationship.** Not as a daily
one: the 1-day change correlation is **−0.079**, and this is not a
daily-rebalance hedge. The evidence is four independent things pointing the
same way, which matters because no one of them is strong enough alone on a
503-day sample:

1. levels **+0.713**, same sign in every year, 93% of rolling windows;
2. the horizon profile **−0.079 → +0.175 → +0.207 → +0.480** against the fly's
   flat **−0.283 → +0.025 → −0.012 → +0.008**, confirmed on non-overlapping
   subsamples (+0.51 at 63d, n=7 blocks; +0.27 at 5d, n=64);
3. the **13 × 7 diagonal ridge** — the matching expiry moves out with pack
   rank across thirteen packs and seven nodes, which is structure a single
   spurious correlation cannot produce;
4. the **leg decomposition**, where the pack leg is more vol-sensitive than
   the swap leg by **0.112 bp/bp** against a Ho-Lee prediction of 0.097 — the
   right sign *and* the right size.

**So Citi's economics survive and only their hedge instrument failed.** The
actionable conclusion is to hedge pack convexity **with vol, not with a fly**:

* at the pack's **own matched expiry** × 1Y tenor — 3Y1Y for Blues (rank 13,
  `t1_rms` 3.51y), but 2Y1Y for rank 5 and 4–5Y1Y for rank 17; the node is a
  function of the pack's expiry, not of the colour;
* sized at **`dCA/dvol ≈ 0.11 bp of CA per bp of normal vol`**, which is
  `σ·M/1e4 · dσ/dvol` and can be recomputed per pack per day from
  `time_weight` and the current implied vol, rather than from a 2017 β;
* rebalanced **monthly or slower**. At a daily frequency the leg noise
  (CA 1-day sd 1.74bp against a 0.11 bp/bp signal) swamps it.

**What would refute this and did not:** our CA construction (the S-shape is
unstable in sign and does not track where the link is weak — rank 8 has the
second-largest residual and the best correlation); the expiry mapping (matched,
3Y and 4Y are within 0.012 of each other at Blues); and an absence of vol
signal in the SOFR strip (it is in both legs at R² 0.68–0.71, and the CA keeps
the predicted fraction). Those three are kept apart deliberately, because
blurring them is how "it didn't work" gets mistaken for "it can't work".

In [29]:
SUMMARY = {
    "claim_A_fly_proxy": {
        "verdict": "EXPIRED (restated from citi_fig89_reproduction)",
        "fig8_levels_corr_published_weights": -0.61,
        "fig9_levels_corr_published_weights": -0.62,
        "citi_stated": 0.90,
        "fig9_by_year": [0.68, -0.67, 0.07, -0.01],
        "variance_reduction_pct_citi_beta": -10.5,
    },
    "claim_B_ca_vol_link": {
        "verdict": VERDICT["verdict"],
        "checks": VERDICT["checks"],
        "diagnostics": VERDICT["diagnostics"],
        "headline": {
            "pack_rank": CFG.headline_rank,
            "vol_node": f"{CFG.citi_node}x{CFG.vol_tenor}",
            "corr_levels": float(_hv["corr_levels"]), "n_levels": int(_hv["n_levels"]),
            "corr_by_horizon": {int(h): float(_hv[f"corr_d{h}"]) for h in CFG.horizons},
            "corr_by_horizon_fly": {int(h): float(_hf[f"corr_d{h}"]) for h in CFG.horizons},
            "sign_stable_frac_vol": _sv, "sign_stable_frac_fly": _sf,
            "regime_shift_in_early_resid_sd": _rs,
        },
        "mechanism": {
            "d63_beta_pack_leg_on_vol": _bp,
            "d63_beta_swap_leg_on_vol": _bs,
            "d63_beta_ca_on_vol_measured": _bc,
            "ho_lee_dCA_dsigma": _dCAdsig,
            "measured_dsigma_dvol": float(_dsigdvol),
            "dCA_dvol_predicted": float(_dCAdsig * _dsigdvol),
        },
        "expiry_match": json.loads(EXPMATCH.reset_index().to_json(orient="records")),
        "citi_iv_tieout": {"max_abs_bp": _maxiv, "median_abs_bp": _medv, "n_rows": len(TIEOUT)},
        "construction_residual": {
            "corr_absdca_vs_link": _r_dmg,
            "d_ca_by_rank": json.loads(TIEOUT[["rank", "d_ca", "d_iv"]].to_json(orient="records")),
        },
    },
    "caveats": {
        "blues_pack_days": int(len(BLUES)),
        # first/last is NOT the sample: 2023-26 contribute 49/19/2/1 days. The
        # effective window is where the density is, and quoting only first/last
        # would understate the thinnest caveat in the whole notebook.
        "blues_span_first_last": [str(BLUES.index.min().date()),
                                  str(BLUES.index.max().date())],
        "blues_pack_days_by_year": {int(k): int(v) for k, v in
                                    BLUES.groupby(BLUES.index.year).size().items()},
        "blues_effective_span": "2021-01 .. 2023-mid (90% of pack-days fall before "
                                + str(BLUES.index[int(0.9 * len(BLUES))].date())
                                + "); the tail is 1-49 days a year",
        "independent_blocks_at_d63": round(float(_hv["n_d63"]) / 63.0, 1),
        "iv_over_vol_median": float((_sc["iv"] / _sc["vol"]).median()),
        "model_sigma_is_fitted_not_calibrated": True,
        "test_iii_uses_model_sigma": False,
    },
}
(DATA / "ca_vol_link_verdict.json").write_text(json.dumps(SUMMARY, indent=2, default=str))
print(json.dumps(SUMMARY["claim_B_ca_vol_link"]["headline"], indent=2, default=str))
print(json.dumps(SUMMARY["claim_B_ca_vol_link"]["mechanism"], indent=2, default=str))
print(f"\nwrote {DATA / 'ca_vol_link_verdict.json'}")

{
  "pack_rank": 13,
  "vol_node": "3Yx1Y",
  "corr_levels": 0.6080013018898206,
  "n_levels": 595,
  "corr_by_horizon": {
    "1": -0.08917887495033167,
    "5": 0.12252527223887154,
    "21": 0.1197642896357111,
    "63": 0.41207148203628563
  },
  "corr_by_horizon_fly": {
    "1": -0.1697823398587976,
    "5": 0.02365177657649873,
    "21": 0.012518435681340008,
    "63": 0.020975187012105975
  },
  "sign_stable_frac_vol": 0.9263392857142857,
  "sign_stable_frac_fly": 0.38,
  "regime_shift_in_early_resid_sd": -0.892135930950229
}
{
  "d63_beta_pack_leg_on_vol": 2.1149161333156274,
  "d63_beta_swap_leg_on_vol": 2.0117964445429375,
  "d63_beta_ca_on_vol_measured": 0.10311968877269043,
  "ho_lee_dCA_dsigma": 0.14449533652610821,
  "measured_dsigma_dvol": 0.5628982353659473,
  "dCA_dvol_predicted": 0.08133616994915502
}

wrote C:\Users\chris\clee\ARBS-cvx\notebooks\data\convexity_rv\ca_vol_link_verdict.json
